# Spatial ssGSEA and Immune Spatial Niche Neighborhood Clustering

This Notebook performs gene set scoring on the GBM cohort Visium data. A pre-made multi-sample meta-control is loaded for non-malignant comparison, a healthy average of each gene is then computed across all meta-conrol spots. Each gene in all cohoort Visium spots is z-scored against the healthy average of the meta-control. 

Multiple options are given for gene set dictionaries. ssGSEA is performed on selected single samples (or the entire cohort) for every gene set in the provided dictionary. Normalized Enrichment Scores (NES) for each gene set in each spot is saved to the corresponding .csv for each sample.

In [ ]:
import os
import subprocess
import time
import warnings
from pathlib import Path

import polars as pl
import decoupler as dc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scanpy as sc
from typing import Dict, List, Set, Tuple
from matplotlib.patches import Patch
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist

from gbmcollaboration import AllDataCenters
from gbmcollaboration.utils import (
    gene_filter_by_expression,
    # load_visium_adata,
    plot_gene_filtering_thresholds,
    load_gene_sets,
    score_gene_sets,
)

def get_data(adata, layer=None):
    if layer is not None:
        return adata.layers[layer]
    else:
        return adata.X

def split_dict_into_batches(d, batch_size):
    items = list(d.items())
    return [dict(items[i:i + batch_size]) for i in range(0, len(items), batch_size)]

### Parameters

In [ ]:
BATCH_KEY = 'sample'
LAYER = "log_norm_count" # None for adata.X
METACONTROL_PATH = Path("/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/meta_control_v2.h5ad")
RESULTS_DIR = Path("/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results/visium/spatial_gsea")
GENE_SETS_DIR = RESULTS_DIR / 'gene_sets'
GENE_SETS_URLS = {
    "hallmark": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2024.1.Hs/h.all.v2024.1.Hs.symbols.gmt",
    "biocarta": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2024.1.Hs/c2.cp.biocarta.v2024.1.Hs.symbols.gmt",
    "wikipathways": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2024.1.Hs/c2.cp.wikipathways.v2024.1.Hs.symbols.gmt",
    "immunesigdb": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2024.1.Hs/c7.immunesigdb.v2024.1.Hs.symbols.gmt",
    "cgp": "https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2024.1.Hs/c2.cgp.v2024.1.Hs.symbols.gmt",
}
GENE_SETS_FILES = {
    "immune": GENE_SETS_DIR / "Gene_sets_immune.csv",
    "not_immune": GENE_SETS_DIR / "Gene_sets_not_immune.csv"
}
GENE_MIN_N_SPOTS = 5
GENE_MIN_N_SAMPLES = 2
MIN_GENE_SET_SIZE = 15
MAX_GENE_SET_SIZE = 500
DECONVOLUTION_SLOT_NAME = "deconv_cohort_level_4"

### Load Visium GBM Cohort Data

In [ ]:
adata_cohort = AllDataCenters.load_spatial_adata(
    exclude_low_quality_visium=True,
    exclude_idh1_mut=True,
    list_sample_id=None,
    normalization=LAYER,
    radius=1,
    deconvolution_slot_name=DECONVOLUTION_SLOT_NAME,
)

### Load Meta-Control Sample and Create Log_Norm Layer

In [ ]:
# Load meta-control and reassign log-normalized layer

adata_ctrl = sc.read_h5ad(METACONTROL_PATH)
print(f"Loaded adata_ctrl from {METACONTROL_PATH} → shape: {adata_ctrl.shape}")

# Rename the normalization layer
if "log1p_TP10K" not in adata_ctrl.layers:
    raise KeyError("Error 'log1p_TP10K' not found in adata_ctrl.layers. Please ensure the original normalization step was run.")

adata_ctrl.layers[LAYER] = adata_ctrl.layers["log1p_TP10K"]
print(f"Reassigned 'log1p_TP10K' → '{LAYER}'")

### Remove genes with NAN values and align gene names between control and cohort objects

In [ ]:
plot = False
qc_gene_filter = True

# Align genes between adata_ctrl and adata_cohort. 
common_genes = adata_cohort.var_names.intersection(adata_ctrl.var_names)
adata_cohort = adata_cohort[:, common_genes]
adata_ctrl = adata_ctrl[:, common_genes]
print(f"Aligned gene space: {len(common_genes)} genes shared between control and cohort")

# Drop NaN genes
X_ctrl = get_data(adata_ctrl,layer=LAYER)
X_cohort = get_data(adata_cohort,layer=LAYER)

gene_hasnan_ctrl = np.isnan(X_ctrl).any(0)
gene_hasnan_cohort =  np.isnan(X_cohort).any(0)
genes_to_drop = adata_cohort.var_names[gene_hasnan_cohort | gene_hasnan_ctrl]

common_genes = common_genes.difference(genes_to_drop)
adata_cohort._inplace_subset_var(common_genes)
adata_ctrl._inplace_subset_var(common_genes)
print(f"Dropped nan genes:",*genes_to_drop)

# Choose genes to filter
if qc_gene_filter:
    n_spots_dfs, keep_genes = gene_filter_by_expression(
        adata_case=adata_cohort,
        adata_ctrl=None,
        layer = LAYER,
        min_spots = GENE_MIN_N_SPOTS,
        min_samples = GENE_MIN_N_SAMPLES)

    # Filter genes
    adata_cohort._inplace_subset_var(keep_genes)
    adata_ctrl._inplace_subset_var(keep_genes)

    n_genes = len(common_genes)
    n_genes_retained = len(keep_genes)
    print(f'Retaining {n_genes_retained} out of {n_genes}')


if plot:
    plot_gene_filtering_thresholds(
        n_spots_dfs,
        max_min_spots = 20,
        max_min_samples = None)


### Load gene sets

In [ ]:
gene_sets, excluded_gene_sets = load_gene_sets(
    gene_sets_urls=None,
    gene_sets_files=GENE_SETS_FILES,
    gene_sets_dir=GENE_SETS_DIR,
    valid_genes=adata_cohort.var_names,
    min_gene_set_size=MIN_GENE_SET_SIZE,
    max_gene_set_size=MAX_GENE_SET_SIZE,
    plot=True,
    col_genes='gene',
    col_pathway='gs_name'
    )

### Score gene sets

In [ ]:
# USER PARAMETERS
SAMPLE_ID = None # 'CH_G_196a_vis'
SCORING_METHOD = 'ulm' #'ssgsea'
METACONTROL_ZSCORE = True
GENE_SETS = gene_sets.set_index('gs_name')['gene'].to_dict()
GENE_SETS_SCORE_DIR = RESULTS_DIR / f'{SCORING_METHOD}_metacontrol_zscore={METACONTROL_ZSCORE}_wide'
GENE_SETS_BATCH_SIZE = 10_000
SAMPLE_BATCH_SIZE = 20_000
DECOUPLER_STORAGE = 'wide'

if SAMPLE_ID is not None:
    GENE_SETS_SCORE_DIR = GENE_SETS_SCORE_DIR.with_name(GENE_SETS_SCORE_DIR.name + f'_{SAMPLE_ID}')
    adata_ = adata_cohort[adata_cohort.obs[BATCH_KEY]==SAMPLE_ID]
else:
    adata_ = adata_cohort

# --- STEP 1: VALIDATE DATA ALIGNMENT ---
assert adata_.var_names.equals(adata_ctrl.var_names)

# --- STEP 2: (OPTIONAL) PRE-COMPUTE CONTROL STATISTICS ---
if METACONTROL_ZSCORE:
    print("\nComputing meta-control statistics...")
    ctrl_mat = adata_ctrl.layers[LAYER]
    ctrl_mu = np.nanmean(ctrl_mat, axis=0)
    ctrl_sigma = np.nanstd(ctrl_mat, axis=0)

    zero_std_mask = (ctrl_sigma == 0)
    if np.any(zero_std_mask):
        min_nonzero_std = np.min(ctrl_sigma[~zero_std_mask])
        print(f"  -> Found {np.sum(zero_std_mask)} genes with zero std dev in control.")
        print(f"  -> Replacing with min non-zero std: {min_nonzero_std:.4g}")
        ctrl_sigma[zero_std_mask] = min_nonzero_std

    if LAYER is not None: 
        adata_.layers[LAYER] = (adata_.layers[LAYER] - ctrl_mu) / ctrl_sigma
    else:
        adata_.X = (adata_.X - ctrl_mu) / ctrl_sigma

# # --- STEP 3: COMPUTE SCORES ACROSS SAMPLES ---
kwargs = dict(
    adata=adata_,
    layer=LAYER,
    method=SCORING_METHOD,
    min_size=MIN_GENE_SET_SIZE,
    max_size=MAX_GENE_SET_SIZE,
    verbose=True,
    decoupler_storage=DECOUPLER_STORAGE
)

if SCORING_METHOD != 'ssgsea':
    kwargs['batch_size'] = SAMPLE_BATCH_SIZE # 'bsize' in newer decoupler versions

# Process gene sets by batch and save as separate files
GENE_SETS_SCORE_DIR.mkdir(exist_ok=True,parents=True)

if DECOUPLER_STORAGE == 'wide':
    kwargs['gene_sets'] = GENE_SETS
    kwargs['out_file'] = GENE_SETS_SCORE_DIR / ".parquet"
    _ = score_gene_sets(**kwargs)
else:
    gene_sets_batches = split_dict_into_batches(GENE_SETS, batch_size=GENE_SETS_BATCH_SIZE)
    for i, gene_sets_batch in enumerate(gene_sets_batches):
        kwargs['gene_sets'] = gene_sets_batch
        kwargs['out_file'] = GENE_SETS_SCORE_DIR / f"{i}.parquet"

        if kwargs['out_file'].exists():
            print("Found ",kwargs['out_file'],"skipping")
        else:
            _ = score_gene_sets(**kwargs)